# Extract GFv1.1 Geofabric from HyTest OSN

Download the GFv1.1 HRU fabric (nhru_v1_1) from the HyTest S3 catalog
and save locally as a GeoPackage for use with `nhf-targets init`.

Source: [HyTest intake catalog](https://github.com/hytest-org/hytest/blob/main/dataset_catalog/hytest_intake_catalog.yml)
DOI: https://doi.org/10.5066/P971JAGF

In [ ]:
import geopandas as gpd
import urllib.request
from pathlib import Path


In [5]:
# HyTest OSN endpoint and GFv1.1 layers
BASE_URL = "https://usgs.osn.mghpcc.org/hytest/geofabric_v1_1"

LAYERS = {
    "nhru_v1_1": "GFv1.1_nhru_v1_1.geoparquet",
    "nsegment_v1_1": "GFv1.1_nsegment_v1_1.geoparquet",
    "POIs_v1_1": "GFv1.1_POIs_v1_1.geoparquet",
}

# Output directory — change as needed
OUT_DIR = Path("../data/geofabric")
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [6]:
# Download each layer, then read locally
# (direct HTTP streaming triggers a snappy decompression bug in pyarrow)
gdfs = {}
for name, filename in LAYERS.items():
    url = f"{BASE_URL}/{filename}"
    local = OUT_DIR / filename
    if not local.exists():
        print(f"Downloading {name} ...")
        urllib.request.urlretrieve(url, local)
        print(f"  saved to {local} ({local.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"{name} already downloaded ({local.stat().st_size / 1e6:.1f} MB)")
    gdfs[name] = gpd.read_parquet(local)
    print(f"  {gdfs[name].shape[0]:,} features, CRS={gdfs[name].crs}")


  saved to ../data/geofabric/GFv1.1_nhru_v1_1.geoparquet (2626.0 MB)
  114,958 features, CRS={"$schema": "https://proj.org/schemas/v0.5/projjson.schema.json", "type": "ProjectedCRS", "name": "USA_Contiguous_Albers_Equal_Area_Conic_USGS_version", "base_crs": {"name": "NAD83", "datum": {"type": "GeodeticReferenceFrame", "name": "North American Datum 1983", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "id": {"authority": "EPSG", "code": 4269}}, "conversion": {"name": "unnamed", "method": {"name": "Albers Equal Area", "id": {"authority": "EPSG", "code": 9822}}, "parameters": [{"name": "Latitude of false origin", "value": 23, "unit": "degree", "id": {"authority": "EPSG", "code": 8821}}, {"name

In [7]:
# Inspect the HRU layer
nhru = gdfs["nhru_v1_1"]
print(f"Columns: {nhru.columns.tolist()}")
print(f"CRS: {nhru.crs}")
print(f"Bounds: {nhru.total_bounds}")
nhru.head()


Columns: ['nhru_v1_1', 'hru_segment_v1_1', 'nhm_id', 'hru_id_nat', 'Version', 'Shape_Length', 'Shape_Area', 'geometry']
CRS: {"$schema": "https://proj.org/schemas/v0.5/projjson.schema.json", "type": "ProjectedCRS", "name": "USA_Contiguous_Albers_Equal_Area_Conic_USGS_version", "base_crs": {"name": "NAD83", "datum": {"type": "GeodeticReferenceFrame", "name": "North American Datum 1983", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "id": {"authority": "EPSG", "code": 4269}}, "conversion": {"name": "unnamed", "method": {"name": "Albers Equal Area", "id": {"authority": "EPSG", "code": 9822}}, "parameters": [{"name": "Latitude of false origin", "value": 23, "unit": "degree", "id": {"authority"

,nhru_v1_1,hru_segment_v1_1,nhm_id,hru_id_nat,Version,Shape_Length,Shape_Area,geometry
0,76127,40038,76128,76128,1,92601.509382,1.881513e+08,"MULTIPOLYGON (((-105544.567 804074.976, -10554..."
1,76147,40038,76148,76148,1,60460.683381,4.416189e+07,"MULTIPOLYGON (((-97185.217 806355.005, -97184...."
2,76170,40021,76171,76171,1,62333.253245,7.337575e+07,"MULTIPOLYGON (((-106590.453 815804.909, -10646..."
3,76172,40019,76173,76173,1,36798.115971,3.627274e+07,"MULTIPOLYGON (((-106762.641 815925.02, -106754..."
4,76181,40019,76182,76182,1,31441.172667,1.564925e+07,"MULTIPOLYGON (((-109785.311 816675.047, -10978..."


In [8]:
# Save each layer as a GeoPackage
for name, gdf in gdfs.items():
    out_path = OUT_DIR / f"{name}.gpkg"
    gdf.to_file(out_path, driver="GPKG")
    print(f"Saved {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)")


ERROR 1: PROJ: proj_create_from_database: Open of /home/rmcd/projects/usgs/nhf-spatial-targets/.pixi/envs/dev/share/proj failed


Saved ../data/geofabric/nhru_v1_1.gpkg  (3351.0 MB)
Saved ../data/geofabric/nsegment_v1_1.gpkg  (416.2 MB)
Saved ../data/geofabric/POIs_v1_1.gpkg  (9.3 MB)
